# 04 — Test risk-engine (WebSocket bridge end-to-end)

Smoke test du container `fxvol-risk` — étape 4/5. Valide la **chaîne complète** depuis l'engine jusqu'à un client WebSocket type frontend :

```
engine risk-engine (cycle 2s)
    │ publish_risk_update → channel risk_update (Redis pub/sub)
    ▼
api/ws/redis_bridge.py — SUBSCRIBE risk_update → broadcast WS
    │ forward via /ws/risk
    ▼
nginx — proxy_pass WebSocket sur /ws/ → api:8000
    ▼
Ce notebook (websocket-client) — connect ws://localhost/ws/risk
```

Si ce test passe → **`useRiskStream` côté frontend est validé en amont**, prêt pour quand le smoke frontend arrivera.

**Couvre** :
1. Seed surface (sinon cycle skip)
2. Container `fxvol-api` running + healthy (gate)
3. Container `fxvol-nginx` running + healthy (gate)
4. Connect `ws://localhost/ws/risk` réussit
5. Réception ≥ 2 messages JSON valides en 5s
6. Schéma identique au pub/sub Redis : `{timestamp, greeks:{delta, gamma, vega, theta, spot}}`

**Préreq** :
- Notebooks 01-03 verts
- Stack complète : `docker compose --profile engines --profile ib up -d`
- `pip install websocket-client`

## Setup + seed surface

In [1]:
import json
import math
import subprocess
import time
from datetime import datetime, timezone

import redis

try:
    import websocket
except ImportError:
    raise RuntimeError("package `websocket-client` non installé.\n  -> python -m pip install websocket-client")

WS_URL = "ws://localhost/ws/risk"
REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"
EXPECTED_GREEKS_KEYS = {"delta", "gamma", "vega", "theta", "spot"}

MIN_MESSAGES = 2
LISTEN_DURATION_S = 5.0
RECV_TIMEOUT_S = 1.0

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(container, fmt):
    out = subprocess.run(["docker", "inspect", "-f", fmt, container],
                         capture_output=True, text=True)
    return out.stdout.strip()

# Seed surface inline (cf. notebook 02 §1)
r = redis.from_url(REDIS_URL, decode_responses=True)
fake_surface = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "surface": {"1M": {"atm": {"iv": 0.07, "strike": 1.17}}},
}
r.set(f"latest_vol_surface:{SYMBOL}", json.dumps(fake_surface), ex=600)
print(f"target = {WS_URL}, surface seedé")

target = ws://localhost/ws/risk, surface seedé


## 1. Gate — `fxvol-api` running + healthy

Le bridge Redis→WS vit dans le container api. Si api est down, `/ws/risk` ne répond pas.

In [2]:
print("== 1. fxvol-api running + healthy ==")

state = docker_inspect("fxvol-api", "{{.State.Status}}")
record("fxvol-api state running", state == "running", state or "<not found>")

health = docker_inspect("fxvol-api", "{{.State.Health.Status}}")
record("fxvol-api healthcheck", health == "healthy", health or "<no healthcheck>")

== 1. fxvol-api running + healthy ==
  [OK  ] fxvol-api state running  | running
  [OK  ] fxvol-api healthcheck  | healthy


## 2. Gate — `fxvol-nginx` running + healthy

Nginx proxy `/ws/` vers api. Si nginx down, on ne peut pas joindre l'api depuis l'host.

In [3]:
print("== 2. fxvol-nginx running + healthy ==")

state = docker_inspect("fxvol-nginx", "{{.State.Status}}")
record("fxvol-nginx state running", state == "running", state or "<not found>")

health = docker_inspect("fxvol-nginx", "{{.State.Health.Status}}")
record("fxvol-nginx healthcheck", health == "healthy", health or "<no healthcheck>")

== 2. fxvol-nginx running + healthy ==
  [OK  ] fxvol-nginx state running  | running
  [OK  ] fxvol-nginx healthcheck  | healthy


## 3. Connect WebSocket `/ws/risk`

In [4]:
print("== 3. WebSocket connect ==")

ws = None
try:
    ws = websocket.create_connection(WS_URL, timeout=RECV_TIMEOUT_S)
    record("connect ws://localhost/ws/risk", ws.connected,
           f"connected = {ws.connected}")
except Exception as e:
    record("connect ws://localhost/ws/risk", False, f"{type(e).__name__}: {e}")

== 3. WebSocket connect ==
  [OK  ] connect ws://localhost/ws/risk  | connected = True


## 4. Réception ≥ 2 messages en 5s

Cycle 2s = 2-3 messages max théorique sur 5s, idem notebook 03.

In [5]:
print(f"== 4. réception {LISTEN_DURATION_S}s ==")

raw_messages = []
if ws and ws.connected:
    deadline = time.perf_counter() + LISTEN_DURATION_S
    while time.perf_counter() < deadline:
        try:
            data = ws.recv()
            if data:
                raw_messages.append(data)
        except websocket.WebSocketTimeoutException:
            continue
        except Exception as e:
            print(f"  [WARN] WS recv error: {type(e).__name__}: {e}")
            break

record(f"≥ {MIN_MESSAGES} messages reçus en {LISTEN_DURATION_S}s",
       len(raw_messages) >= MIN_MESSAGES,
       f"{len(raw_messages)} messages")

== 4. réception 5.0s ==
  [OK  ] ≥ 2 messages reçus en 5.0s  | 3 messages


## 5. Schéma + types — identique au pub/sub Redis

Le bridge ne transforme pas le payload. Format attendu : `{timestamp, greeks:{delta, gamma, vega, theta, spot}}`.

In [6]:
print("== 5. schéma + types ==")

if not raw_messages:
    record("schéma + types", False, "skip (cf. §4)")
else:
    parse_errors = 0
    schema_errors = 0
    type_errors = 0
    sample = None
    for raw in raw_messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            parse_errors += 1
            continue
        if not ({"timestamp", "greeks"} <= set(obj.keys())):
            schema_errors += 1
            continue
        if not (EXPECTED_GREEKS_KEYS <= set(obj.get("greeks", {}).keys())):
            schema_errors += 1
            continue
        if sample is None:
            sample = obj
        # types
        for k in EXPECTED_GREEKS_KEYS:
            v = obj["greeks"].get(k)
            if not isinstance(v, (int, float)) or (isinstance(v, float) and math.isnan(v)):
                type_errors += 1
                break
    record("tous JSON-parseables",
           parse_errors == 0,
           f"{parse_errors} erreurs / {len(raw_messages)}")
    record("top-level + 5 keys greeks",
           schema_errors == 0,
           f"{schema_errors} schémas incomplets / {len(raw_messages)}")
    record("greeks values floats finis",
           type_errors == 0,
           f"{type_errors} type errors / {len(raw_messages)}")
    if sample:
        print(f"  [INFO] sample = {sample}")

== 5. schéma + types ==
  [OK  ] tous JSON-parseables  | 0 erreurs / 3
  [OK  ] top-level + 5 keys greeks  | 0 schémas incomplets / 3
  [OK  ] greeks values floats finis  | 0 type errors / 3
  [INFO] sample = {'timestamp': '2026-04-28T15:25:25.966019Z', 'greeks': {'delta': 0.0, 'gamma': 0.0, 'vega': 0.0, 'theta': 0.0, 'spot': 1.170925}}


## Cleanup

In [7]:
if ws:
    try:
        ws.close()
        print("WebSocket closed cleanly.")
    except Exception as e:
        print(f"WS close warning: {type(e).__name__}: {e}")

WebSocket closed cleanly.


## Récap final

In [8]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — chaîne risk-engine → Redis → api → nginx → WS validée bout-à-bout.")
    print("Pass au notebook 05 (resilience) puis surface risk-engine complétée.")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
fxvol-api state running                                 OK      running
fxvol-api healthcheck                                   OK      healthy
fxvol-nginx state running                               OK      running
fxvol-nginx healthcheck                                 OK      healthy
connect ws://localhost/ws/risk                          OK      connected = True
≥ 2 messages reçus en 5.0s                              OK      3 messages
tous JSON-parseables                                    OK      0 erreurs / 3
top-level + 5 keys greeks                               OK      0 schémas incomplets / 3
greeks values floats finis                              OK      0 type errors / 3
--------------------------------------------------------------------------------------------------------------

9 OK / 0 FAI

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `fxvol-api unhealthy` | Postgres/Redis pas joignables, alembic plante | `docker logs fxvol-api --tail 50` |
| `fxvol-nginx unhealthy` | api upstream pas joignable | `docker compose restart nginx` |
| `connect WS ConnectionError` | nginx/api down | `docker compose ps` |
| `connect OK + 0 message` | bridge `redis_bridge.py` pas SUBSCRIBE à `risk_update` | grep `CH_RISK_UPDATE` dans `src/api/ws/redis_bridge.py:_FORWARDED` |
| `schema_errors > 0` | bridge transforme le payload | inspecter `redis_bridge.py:_serve` |
| `greeks values NaN` | bug calcul engine | déjà couvert au notebook 02 §4 — si fail seulement ici, suspect bridge |